# Synthetic Tabular Data Generation - Staged Optimization Driver

**Enhanced notebook for synthetic tabular data generation with staged hyperparameter optimization.**

This notebook provides a staged optimization workflow with:
- Section 1: Setup and imports
- Section 2: Data loading, preprocessing, and EDA
- Section 3: Demo models with default parameters
- Section 4: **Staged Hyperparameter Optimization** (NEW)
  - 4.1: Configuration
  - 4.2: Pilot phase (15 trials) with time estimates
  - 4.3: Continuation options
  - 4.4: Diminishing returns analysis
  - 4.5: Additional batches (optional)
- Section 5: Final models with best parameters

Based on STG-Driver-breast-cancer.ipynb with staged optimization enhancements.

## 1 Setup and Configuration

In [1]:
# Code Chunk ID: CHUNK_1_0_0_A - Import Setup Module
# Import all functionality from setup.py
import sys
sys.path.insert(0, "/home/ec2-user/SageMaker/tableGenCompare")

from setup import *

# Refresh session timestamp to current date
refresh_session_timestamp()

print("SETUP MODULE IMPORTED SUCCESSFULLY!")
print("="*60)
import sys                                                                                                                                                                                                     
!{sys.executable} -m pip install ctgan sdv ganeraid optuna -q

[OK] Essential data science libraries imported successfully!
[OK] Model registry loaded from src/models/registry.py
[OK] Batch training module loaded from src/models/batch_training.py
[OK] Search spaces module loaded from src/models/search_spaces.py
[OK] Batch optimization module loaded from src/models/batch_optimization.py
[OK] Staged optimization module loaded from src/models/staged_optimization.py
[CONFIG] Session timestamp: 2026-02-19
[OK] Parameter management functions loaded from src/utils/parameters.py
[OK] Hyperparameter optimization evaluation functions loaded from src/evaluation/hyperparameters.py
[OK] Optuna objective functions for PATE-GAN and MEDGAN loaded (Phase 5)
Detected sklearn 1.7.2 - applying compatibility patch...
INFO: Applying sklearn compatibility patches for version 1.7.2
Global sklearn compatibility patch applied successfully
CTAB-GAN imported successfully
[OK] CTAB-GAN+ detected and available
[OK] GANerAidModel imported successfully from src.models.implementa

## 2 Data Loading and Pre-processing

### 2.1 Configuration

**USER ATTENTION NEEDED**: Edit the `NOTEBOOK_CONFIG` dictionary below to match your dataset.

In [8]:
# Code Chunk ID: CHUNK_2_1_1_A - NOTEBOOK CONFIGURATION
# ============================================================================
# USER CONFIGURATION - Edit ONLY this block for your dataset
# ============================================================================

NOTEBOOK_CONFIG = {
    # ========== REQUIRED: Dataset Settings ==========
    "data_file": "data/alzheimers_disease_data_processed.csv",  # Path to your CSV file
    "target_column": "Diagnosis",  # Target/outcome column name

    # ========== OPTIONAL: Dataset Metadata ==========
    "dataset_name": "Alzheimer's Dataset",  # Display name
    "dataset_identifier_override": None,  # Override auto-detected ID (or None)

    # ========== OPTIONAL: Column Configuration ==========
    "categorical_columns": [
        "Gender",
        "Ethnicity",
        "EducationLevel",
        "Smoking",
        "FamilyHistoryAlzheimers",
        "CardiovascularDisease",
        "Diabetes",
        "Depression",
        "HeadInjury",
        "Hypertension",
        "MemoryComplaints",
        "BehavioralProblems",
        "Confusion",
        "Disorientation",
        "PersonalityChanges",
        "DifficultyCompletingTasks",
        "Forgetfulness",
    ],
    "task_type": "auto",  # "auto" | "classification" | "regression"

    # ========== OPTIONAL: Data Subsetting ==========
    "use_row_subset": False,  # True to sample rows for faster testing
    "sample_n": 500,  # Number of rows to sample
    "sample_random_state": 42,  # Random seed for reproducibility

    # ========== OPTIONAL: Missing Data Handling ==========
    "missing_strategy": "none",  # "none" | "drop" | "median" | "mode" | "mice" | "indicator_onehot"
    "mice_max_iter": 10,  # Max iterations for MICE imputation

    # ========== OPTIONAL: Model Selection ==========
    "models_to_run": "all",  # "all" or list like ["ctgan", "tvae"]
    # "models_to_run": ["ctgan", "tvae", "copulagan"],

    # ========== OPTIONAL: Tuning Configuration (for Section 3 demo) ==========
    "tuning_mode": "smoke",  # "smoke" (quick) | "full" (thorough)
    "n_trials_smoke": 5,  # Trials for smoke testing
    "n_trials_full": 30,  # Trials for full optimization
}

print("NOTEBOOK_CONFIG set successfully!")
print(f"  Data file: {NOTEBOOK_CONFIG['data_file']}")
print(f"  Target column: {NOTEBOOK_CONFIG['target_column']}")
print(f"  Models to run: {NOTEBOOK_CONFIG['models_to_run']}")
print(f"  Tuning mode: {NOTEBOOK_CONFIG['tuning_mode']}")


NOTEBOOK_CONFIG set successfully!
  Data file: data/alzheimers_disease_data_processed.csv
  Target column: Diagnosis
  Models to run: all
  Tuning mode: smoke


### 2.2 Load and Preprocess Data

In [9]:
# Code Chunk ID: CHUNK_2_1_2_A - LOAD AND PREPROCESS DATA
# ============================================================================
# This uses the config-driven preprocessing pipeline
# ============================================================================

# Load and preprocess data using the config
(
    data,                  # Processed DataFrame
    original_data,         # Copy for reference
    target_column,         # Target column name (cleaned)
    DATASET_IDENTIFIER,    # Dataset identifier for results paths
    categorical_columns,   # List of categorical columns
    metadata               # Full preprocessing metadata
) = load_and_preprocess_from_config(NOTEBOOK_CONFIG)

# Set aliases for backward compatibility with existing notebook cells
TARGET_COLUMN = target_column

# Get models to run based on config
models_to_run = get_models_to_run(NOTEBOOK_CONFIG)
tuning_config = get_tuning_config(NOTEBOOK_CONFIG)
N_TRIALS = get_n_trials(NOTEBOOK_CONFIG)

# Set RUN_MODE for backward compatibility with Section 4 tuning cells
RUN_MODE = "debug" if NOTEBOOK_CONFIG['tuning_mode'] == "smoke" else "full"

print()
print("=" * 60)
print("PREPROCESSING COMPLETE")
print("=" * 60)
print(f"  Dataset identifier: {DATASET_IDENTIFIER}")
print(f"  Processed shape: {data.shape}")
print(f"  Target column: {target_column}")
print(f"  Task type: {metadata['task_type']}")
print(f"  Categorical columns: {categorical_columns}")
print(f"  Models to run: {models_to_run}")
print(f"  RUN_MODE: {RUN_MODE}")
print(f"  N_TRIALS: {N_TRIALS}")
print(f"  Session: {SESSION_TIMESTAMP}")
print(f"  Results path: {get_results_path(DATASET_IDENTIFIER, 2)}")
print("=" * 60)

[LOAD] Loading data from: data/alzheimers_disease_data_processed.csv
[LOAD] Loaded 2149 rows, 33 columns
[PREPROCESS] Starting preprocessing pipeline
[PREPROCESS] Input shape: (2149, 33)
[PREPROCESS] Categorical columns: ['gender', 'ethnicity', 'educationlevel', 'smoking', 'familyhistoryalzheimers', 'cardiovasculardisease', 'diabetes', 'depression', 'headinjury', 'hypertension', 'memorycomplaints', 'behavioralproblems', 'confusion', 'disorientation', 'personalitychanges', 'difficultycompletingtasks', 'forgetfulness']
[PREPROCESS] Task type: classification
[PREPROCESS] Final shape: (2149, 33)
[PREPROCESS] Dataset identifier: alzheimers-disease-data-processed

PREPROCESSING COMPLETE
  Dataset identifier: alzheimers-disease-data-processed
  Processed shape: (2149, 33)
  Target column: diagnosis
  Task type: classification
  Categorical columns: ['gender', 'ethnicity', 'educationlevel', 'smoking', 'familyhistoryalzheimers', 'cardiovasculardisease', 'diabetes', 'depression', 'headinjury', '

### 2.3 Exploratory Data Analysis (EDA)

Comprehensive EDA with automated file export to results directory.

In [10]:
# Code Chunk ID: CHUNK_2_1_EDA - COMPREHENSIVE EDA
# ============================================================================
# Run comprehensive EDA with single function call
# ============================================================================

from src.data.eda import run_comprehensive_eda

eda_results = run_comprehensive_eda(
    data=data,
    target_column=target_column,
    dataset_identifier=DATASET_IDENTIFIER,
    dataset_name=NOTEBOOK_CONFIG.get('dataset_name'),
    categorical_columns=categorical_columns,
    verbose=True
)

# Update categorical_columns from EDA results (in case auto-detected)
categorical_columns = eda_results['categorical_columns']

print(f"\nEDA Results Summary:")
print(f"  Files generated: {len(eda_results['files_generated'])}")
print(f"  Categorical columns: {categorical_columns}")
print(f"  Balance ratio: {eda_results.get('balance_ratio', 'N/A')}")

COMPREHENSIVE EXPLORATORY DATA ANALYSIS
Dataset: Alzheimer's Dataset
Target column: diagnosis
Results path: results/alzheimers-disease-data-processed/2026-02-19/Section-2

[1/6] Dataset Overview...
   Dataset Name.................. Alzheimer's Dataset
   Shape......................... 2149 rows x 33 columns
   Memory Usage.................. 0.54 MB
   Total Missing Values.......... 0
   Missing Percentage............ 0.00%
   Duplicate Rows................ 0
   Numeric Columns............... 33
   Categorical Columns........... 0

[2/6] Column Analysis...
   Saved: column_analysis.csv (33 columns)

[3/6] Target Variable Analysis...
   Saved: target_analysis.csv
   Saved: target_balance_metrics.csv
   Balance Ratio: 0.547 (Moderately Imbalanced)

[4/6] Feature Distributions...
   Saved: 6 distribution file(s) (32 features)

[5/6] Correlation Analysis...
   Saved: correlation_heatmap.png
   Saved: correlation_matrix.csv
   Saved: target_correlations.csv (32 features)

[6/6] Configuration

## 3 Demo All Models with Default Parameters

### 3.1 Batch Model Training

In [11]:
# Code Chunk ID: CHUNK_3_1_BATCH
# ============================================================================
# SECTION 3.1 - BATCH MODEL TRAINING
# Train all models configured in NOTEBOOK_CONFIG['models_to_run']
# ============================================================================

from src.models.batch_training import train_models_batch, extract_synthetic_data_to_globals

# Run batch training for all configured models
training_results = train_models_batch(
    data=data,
    models_to_run=models_to_run,
    target_column=target_column,
    categorical_columns=categorical_columns,
    n_samples=len(data),
    random_state=42,
    verbose=True
)

# Extract synthetic_data_* variables for Section 3.2 compatibility
created_vars = extract_synthetic_data_to_globals(training_results, globals())
print(f"\nCreated variables: {created_vars}")

# Also create model_* variables for backward compatibility
for model_name, result in training_results.items():
    if result['status'] == 'success':
        globals()[f'{model_name}_model'] = result['model']


BATCH MODEL TRAINING
Models to train: 7
Dataset shape: (2149, 33)
Target column: diagnosis
Samples to generate: 2149


[1/7] Training CTGAN...
--------------------------------------------------
  Training CTGAN...


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml fo

  Generating 2149 synthetic samples...
  [OK] CTGAN completed in 353.78s
  Synthetic data shape: (2149, 33)

[2/7] Training TVAE...
--------------------------------------------------
  Training TVAE...


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml fo

  Generating 2149 synthetic samples...
[OK] Target integrity functions loaded from src/data/target_integrity.py
  [OK] TVAE completed in 69.83s
  Synthetic data shape: (2149, 33)

[3/7] Training CopulaGAN...
--------------------------------------------------
  Training CopulaGAN...
  Generating 2149 synthetic samples...
  [OK] CopulaGAN completed in 340.09s
  Synthetic data shape: (2149, 33)

[4/7] Training CTABGAN...
--------------------------------------------------
  Training CTABGAN...


100%|██████████| 300/300 [02:49<00:00,  1.77it/s]


Finished training in 179.84147453308105  seconds.
  Generating 2149 synthetic samples...
  [OK] CTABGAN completed in 180.07s
  Synthetic data shape: (2149, 33)

[5/7] Training CTABGAN+...
--------------------------------------------------
  Training CTABGAN+...


100%|██████████| 400/400 [03:31<00:00,  1.89it/s]


Finished training in 219.3728153705597  seconds.
  Generating 2149 synthetic samples...
  [OK] CTABGAN+ completed in 219.58s
  Synthetic data shape: (2149, 33)

[6/7] Training PATE-GAN...
--------------------------------------------------
  Training PATE-GAN...
  Generating 2149 synthetic samples...
  [OK] PATE-GAN completed in 12.03s
  Synthetic data shape: (2149, 33)

[7/7] Training MEDGAN...
--------------------------------------------------
  Training MEDGAN...
  Generating 2149 synthetic samples...
  [OK] MEDGAN completed in 32.37s
  Synthetic data shape: (2149, 33)

BATCH TRAINING SUMMARY
Total models: 7
Successful: 7
Failed: 0

Successful models:
  - CTGAN: 353.78s
  - TVAE: 69.83s
  - CopulaGAN: 340.09s
  - CTABGAN: 180.07s
  - CTABGAN+: 219.58s
  - PATE-GAN: 12.03s
  - MEDGAN: 32.37s

Created variables: ['synthetic_data_ctgan', 'synthetic_data_tvae', 'synthetic_data_copulagan', 'synthetic_data_ctabgan', 'synthetic_data_ctabganplus', 'synthetic_data_pategan', 'synthetic_data_me

### 3.2 Batch Evaluation

In [12]:
# Code Chunk ID: CHUNK_3_2_0_A
# ============================================================================
# SECTION 3.2 - BATCH EVALUATION FOR ALL TRAINED MODELS
# ============================================================================

print("SECTION 3.2 - COMPREHENSIVE BATCH EVALUATION")
print("=" * 60)

section3_results = evaluate_all_available_models(
    section_number=3,
    scope=globals(),
    models_to_evaluate=None,
    real_data=None,
    target_col=None
)

if section3_results:
    print(f"\nSECTION 3 BATCH EVALUATION COMPLETED!")
    print(f"Evaluated {len(section3_results)} models successfully")
    
    # Show ranking by quality score
    best_models = []
    for model_name, results in section3_results.items():
        if 'error' not in results:
            quality_score = results.get('overall_quality_score', 0)
            best_models.append((model_name, quality_score))
    
    if best_models:
        best_models.sort(key=lambda x: x[1], reverse=True)
        print(f"\nRANKING BY QUALITY SCORE:")
        for i, (model, score) in enumerate(best_models, 1):
            print(f"   {i}. {model}: {score:.3f}")
else:
    print("\nNo models available for evaluation")

SECTION 3.2 - COMPREHENSIVE BATCH EVALUATION
[OK] Batch evaluation functions loaded from src/evaluation/batch.py
[SEARCH] UNIFIED BATCH EVALUATION - SECTION 3
[INFO] Dataset: alzheimers-disease-data-processed
[INFO] Target column: diagnosis
[INFO] Variable pattern: standard
[INFO] Found 7 trained models:
   [OK] CTGAN
   [OK] CTABGAN
   [OK] CTABGANPLUS
   [OK] CopulaGAN
   [OK] TVAE
   [OK] MEDGAN
   [OK] PATEGAN

==================== EVALUATING CTGAN ====================
[SEARCH] CTGAN - COMPREHENSIVE DATA QUALITY EVALUATION
[FOLDER] Output directory: results/alzheimers-disease-data-processed/2026-02-19/Section-3/CTGAN

[1] STATISTICAL SIMILARITY
------------------------------
   [CHART] Average Statistical Similarity: 0.850

[2] PCA COMPARISON ANALYSIS WITH OUTCOME COLOR-CODING
--------------------------------------------------
   [CHART] PCA comparison plot saved: pca_comparison_with_outcome.png
   [CHART] PCA Overall Similarity: 0.011
   [CHART] Explained Variance (PC1, PC2): 0.05

## 4 Staged Hyperparameter Optimization

This section uses a staged approach to hyperparameter optimization:
1. **Pilot phase**: Run a small number of trials to establish time estimates
2. **User decision**: Choose continuation strategy based on time estimates
3. **Continuation**: Run additional trials with chosen strategy
4. **Analysis**: Assess diminishing returns to decide when to stop

### 4.1 Configuration

Create the `StagedOptimizationManager` with configuration.

In [13]:
# Code Chunk ID: CHUNK_4_1_CONFIG
# ============================================================================
# SECTION 4.1 - STAGED OPTIMIZATION CONFIGURATION
# ============================================================================

from src.models.staged_optimization import (
    StagedOptimizationManager,
    StagedOptimizationConfig
)

# Configure staged optimization
staged_config = StagedOptimizationConfig(
    pilot_trials=10,           # Trials for pilot phase
    verbose_level=1,           # 0=silent, 1=concise, 2=detailed
    persistence_dir=f"results/{DATASET_IDENTIFIER}/optimization_studies",
    run_mode=RUN_MODE,         # "debug" or "full"
    random_state=42,
    continue_on_error=True
)

# Create the manager
manager = StagedOptimizationManager(
    data=data,
    target_column=target_column,
    categorical_columns=categorical_columns,
    dataset_identifier=DATASET_IDENTIFIER,
    config=staged_config
)

print("Staged Optimization Manager created!")
print(f"  Pilot trials: {staged_config.pilot_trials}")
print(f"  Run mode: {staged_config.run_mode}")
print(f"  Persistence dir: {staged_config.persistence_dir}")

Staged Optimization Manager created!
  Pilot trials: 10
  Run mode: debug
  Persistence dir: results/alzheimers-disease-data-processed/optimization_studies


### 4.2 Run Pilot Phase

Run a pilot phase with 15 trials per model to establish time estimates.
After this cell, you'll see:
- Average trial time per model
- Trials per hour estimates
- Projected time for additional trials

In [ ]:
# Code Chunk ID: CHUNK_4_2_PILOT
# ============================================================================
# SECTION 4.2 - RUN PILOT PHASE (15 trials per model)
# ============================================================================

# Run pilot phase
pilot_states = manager.run_pilot(
    models_to_run=models_to_run,
    n_trials=5  # Pilot trials
)

# Display time estimates
print("\n" + "="*60)
print("PILOT PHASE COMPLETE - TIME ESTIMATES")
print("="*60)

time_estimates = manager.get_time_estimates()
if len(time_estimates) > 0:
    print(time_estimates.to_string(index=False))
else:
    print("No time estimates available")

# Show best scores so far
print("\nBest scores after pilot:")
for model_name, state in pilot_states.items():
    print(f"  {model_name}: {state.best_score:.4f} ({state.total_trials} trials)")

[I 2026-02-19 18:24:38,065] A new study created in memory with name: ctgan_hpo_alzheimers-disease-data-processed



STAGED OPTIMIZATION - PILOT PHASE
Models to optimize: 7
Pilot trials per model: 5
Dataset identifier: alzheimers-disease-data-processed


[PILOT] Optimizing CTGAN...
--------------------------------------------------


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml fo

[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 33/33 valid metrics, Average: 0.6484
[OK] TRTS Evaluation: 2 scenarios, Average: 0.6438
[CHART] Combined Score: 0.6466 (Similarity: 0.6484, Accuracy: 0.6438)
[ctgan] Trial 1: Combined Score: 0.6466 (Similarity: 0.6484, Accuracy: 0.6438) Best Score so far: 0.6466


Gen. (-03.69) | Discrim. (-00.07): 100%|██████████| 350/350 [01:49<00:00,  3.19it/s]


...

Finished training in 248.79886603355408  seconds.
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 33/33 valid metrics, Average: 0.6612
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7606
[CHART] Combined Score: 0.7009 (Similarity: 0.6612, Accuracy: 0.7606)
[ctabganplus] Trial 3: Combined Score: 0.7009 (Similarity: 0.6612, Accuracy: 0.7606) Best Score so far: 0.7009


100%|██████████| 400/400 [05:07<00:00,  1.30it/s]


Finished training in 320.8563389778137  seconds.
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 33/33 valid metrics, Average: 0.6491
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7462
[CHART] Combined Score: 0.6879 (Similarity: 0.6491, Accuracy: 0.7462)
[ctabganplus] Trial 4: Combined Score: 0.6879 (Similarity: 0.6491, Accuracy: 0.7462) Best Score so far: 0.7009


100%|██████████| 300/300 [03:36<00:00,  1.38it/s]


Finished training in 226.66149234771729  seconds.
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 33/33 valid metrics, Average: 0.6652
[OK] TRTS Evaluation: 2 scenarios, Average: 0.6759
[CHART] Combined Score: 0.6695 (Similarity: 0.6652, Accuracy: 0.6759)
[ctabganplus] Trial 5: Combined Score: 0.6695 (Similarity: 0.6652, Accuracy: 0.6759) Best Score so far: 0.7009
  [OK] CTABGAN+: 5 trials in 1524.9s
  Best score: 0.7009

[PILOT] Optimizing PATE-GAN...
--------------------------------------------------
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 33/33 valid metrics, Average: 0.5176
[OK] TRTS Evaluation: 2 scenarios, Average: 0.4772
[CHART] Combined Score: 0.5015 (Similarity: 0.5176, Accuracy: 0.4772)
[pategan] Trial 1: Combined Score: 0.5015 (Similarity: 0.5176, Accuracy: 0.4772) Best Score so far: 0.5015
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similari

In [ ]:
# ============================================================================
# DIMINISHING RETURNS ANALYSIS
# ============================================================================
print("DIMINISHING RETURNS ANALYSIS")
print("="*60)

convergence_report = manager.get_diminishing_returns_report()

if len(convergence_report) > 0:
    print(convergence_report.to_string(index=False))
    
    print("\nInterpretation:")
    print("-" * 40)
    for _, row in convergence_report.iterrows():
        print(f"  {row['model']}: {row['recommendation']}")
        if row['has_plateaued']:
            print(f"    -> Model has plateaued at score {row['best_score']:.4f}")
        else:
            print(f"    -> Improvement rate: {row['improvement_rate']:.4f}")
else:
    print("No convergence data available")

DIMINISHING RETURNS ANALYSIS
      model  trials  best_score  improvement_rate  total_improvement  has_plateaued               recommendation
      ctgan       5    0.656896          0.015740           0.010340          False Continue - insufficient data
       tvae       5    0.687278          0.013340           0.009168          False Continue - insufficient data
  copulagan       5    0.613768          0.000000           0.000000           True Continue - insufficient data
    ctabgan       5    0.712713          0.028852           0.020563          False Continue - insufficient data
ctabganplus       5    0.700948          0.009696           0.006796           True Continue - insufficient data
    pategan       5    0.591569          0.152315           0.090105          False Continue - insufficient data
     medgan       5    0.607910          0.000000           0.000000           True Continue - insufficient data

Interpretation:
----------------------------------------
  ctgan: 

### 4.3 Stage-2 Complement to Pilot

Based on the time estimates above, choose ONE of the following continuation strategies:

**Option (i): Common trials** - Same number of additional trials for all models
```python
manager.continue_optimization(additional_trials=30)
```

**Option (ii): Per-model trials** - Different trials per model
```python
manager.continue_optimization(trials_per_model={'ctgan': 50, 'tvae': 30, 'copulagan': 20})
```

**Option (iii): Time-based** - Specify time budget per model in minutes
```python
manager.continue_optimization(time_budget_minutes={'ctgan': 90, 'tvae': 60})
```

**Uncomment and modify ONE option below based on your needs:**

In [16]:
# Code Chunk ID: CHUNK_4_3_CONTINUE
# ============================================================================
# SECTION 4.3 - CONTINUATION (Choose ONE option)
# ============================================================================

# OPTION (i): Common trials for all models (commented out)
continuation_states = manager.continue_optimization(additional_trials=30)

# OPTION (ii): Per-model trials (commented out)
# continuation_states = manager.continue_optimization(
#     trials_per_model={
#         'ctgan': 50,
#         'tvae': 30,
#         'copulagan': 20
#     }
# )

# OPTION (iii): Time-based budget - 15 minutes per model
# Build time budget dict for all models being optimized
# time_budget = {model: 2 for model in models_to_run}
# print(f"Running optimization with 2 minutes per model:")
# for model, minutes in time_budget.items():
#     print(f"  {model}: {minutes} minutes")

# continuation_states = manager.continue_optimization(
#     time_budget_minutes=time_budget
# )

# # Display results
# print("\n" + "="*60)
# print("CONTINUATION COMPLETE")
# print("="*60)
# print(manager.get_best_params_summary().to_string(index=False))


STAGED OPTIMIZATION - CONTINUATION PHASE
  ctgan: 30 additional trials
  tvae: 30 additional trials
  copulagan: 30 additional trials
  ctabgan: 30 additional trials
  ctabganplus: 30 additional trials
  pategan: 30 additional trials
  medgan: 30 additional trials


[CONTINUE] Optimizing CTGAN...
--------------------------------------------------
  Resuming from 5 existing trials


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml fo

[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 33/33 valid metrics, Average: 0.6577
[OK] TRTS Evaluation: 2 scenarios, Average: 0.5819
[CHART] Combined Score: 0.6274 (Similarity: 0.6577, Accuracy: 0.5819)
[ctgan] Trial 6: Combined Score: 0.6274 (Similarity: 0.6577, Accuracy: 0.5819) Best Score so far: 0.6569


Gen. (-03.44) | Discrim. (+00.02): 100%|██████████| 250/250 [01:31<00:00,  2.72it/s]


[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 33/33 valid metrics, Average: 0.6394
[OK] TRTS Evaluation: 2 scenarios, Average: 0.5710
[CHART] Combined Score: 0.6120 (Similarity: 0.6394, Accuracy: 0.5710)
[ctgan] Trial 7: Combined Score: 0.6120 (Similarity: 0.6394, Accuracy: 0.5710) Best Score so far: 0.6569


...

Finished training in 132.766907453537  seconds.
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 33/33 valid metrics, Average: 0.6544
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7427
[CHART] Combined Score: 0.6897 (Similarity: 0.6544, Accuracy: 0.7427)
[ctabganplus] Trial 33: Combined Score: 0.6897 (Similarity: 0.6544, Accuracy: 0.7427) Best Score so far: 0.7248


100%|██████████| 350/350 [02:04<00:00,  2.81it/s]


Finished training in 132.27167081832886  seconds.
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 33/33 valid metrics, Average: 0.6950
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7571
[CHART] Combined Score: 0.7198 (Similarity: 0.6950, Accuracy: 0.7571)
[ctabganplus] Trial 34: Combined Score: 0.7198 (Similarity: 0.6950, Accuracy: 0.7571) Best Score so far: 0.7248


100%|██████████| 350/350 [02:04<00:00,  2.80it/s]


Finished training in 131.47979879379272  seconds.
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 33/33 valid metrics, Average: 0.6715
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7071
[CHART] Combined Score: 0.6857 (Similarity: 0.6715, Accuracy: 0.7071)
[ctabganplus] Trial 35: Combined Score: 0.6857 (Similarity: 0.6715, Accuracy: 0.7071) Best Score so far: 0.7248
  [OK] CTABGAN+: 30 trials in 4765.0s
  Best score: 0.7248

[CONTINUE] Optimizing PATE-GAN...
--------------------------------------------------
  Resuming from 5 existing trials
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 33/33 valid metrics, Average: 0.5295
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7650
[CHART] Combined Score: 0.6237 (Similarity: 0.5295, Accuracy: 0.7650)
[pategan] Trial 6: Combined Score: 0.6237 (Similarity: 0.5295, Accuracy: 0.7650) Best Score so far: 0.6237
[TARGET] Enhanced objective function using t

In [17]:
# ============================================================================
# DIMINISHING RETURNS ANALYSIS
# ============================================================================

print("DIMINISHING RETURNS ANALYSIS")
print("="*60)

convergence_report = manager.get_diminishing_returns_report()

if len(convergence_report) > 0:
    print(convergence_report.to_string(index=False))
    
    print("\nInterpretation:")
    print("-" * 40)
    for _, row in convergence_report.iterrows():
        print(f"  {row['model']}: {row['recommendation']}")
        if row['has_plateaued']:
            print(f"    -> Model has plateaued at score {row['best_score']:.4f}")
        else:
            print(f"    -> Improvement rate: {row['improvement_rate']:.4f}")
else:
    print("No convergence data available")

DIMINISHING RETURNS ANALYSIS
      model  trials  best_score  improvement_rate  total_improvement  has_plateaued         recommendation
      ctgan      35    0.657953               0.0           0.011397           True Stop - plateau reached
       tvae      35    0.708459               0.0           0.030349           True Stop - plateau reached
  copulagan      35    0.660824               0.0           0.047056           True Stop - plateau reached
    ctabgan      35    0.726908               0.0           0.034758           True Stop - plateau reached
ctabganplus      35    0.724843               0.0           0.030691           True Stop - plateau reached
    pategan      35    0.638503               0.0           0.137038           True Stop - plateau reached
     medgan      35    0.625248               0.0           0.017339           True Stop - plateau reached

Interpretation:
----------------------------------------
  ctgan: Stop - plateau reached
    -> Model has plateaue

### 4.5 Additional Batches (Optional)

If the diminishing returns analysis suggests continuing, run additional batches.
You can re-run this cell multiple times.

In [18]:
# Code Chunk ID: CHUNK_4_5_ADDITIONAL
# ============================================================================
# SECTION 4.5 - ADDITIONAL BATCHES (Optional - can repeat)
# ============================================================================

# Skip this cell if you're satisfied with the current results
# Or uncomment and run to add more trials

additional_states = manager.continue_optimization(additional_trials=5)
# 
# print("\nUpdated convergence report:")
# print(manager.get_diminishing_returns_report().to_string(index=False))

print("Cell skipped - uncomment to run additional batches")


STAGED OPTIMIZATION - CONTINUATION PHASE
  ctgan: 5 additional trials
  tvae: 5 additional trials
  copulagan: 5 additional trials
  ctabgan: 5 additional trials
  ctabganplus: 5 additional trials
  pategan: 5 additional trials
  medgan: 5 additional trials


[CONTINUE] Optimizing CTGAN...
--------------------------------------------------
  Resuming from 35 existing trials


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml fo

[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 33/33 valid metrics, Average: 0.6392
[OK] TRTS Evaluation: 2 scenarios, Average: 0.5949
[CHART] Combined Score: 0.6215 (Similarity: 0.6392, Accuracy: 0.5949)
[ctgan] Trial 36: Combined Score: 0.6215 (Similarity: 0.6392, Accuracy: 0.5949) Best Score so far: 0.6580


Gen. (-03.53) | Discrim. (-00.07): 100%|██████████| 250/250 [00:50<00:00,  4.97it/s]


[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 33/33 valid metrics, Average: 0.5994
[OK] TRTS Evaluation: 2 scenarios, Average: 0.5805
[CHART] Combined Score: 0.5919 (Similarity: 0.5994, Accuracy: 0.5805)
[ctgan] Trial 37: Combined Score: 0.5919 (Similarity: 0.5994, Accuracy: 0.5805) Best Score so far: 0.6580


...

Finished training in 112.79796409606934  seconds.
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 33/33 valid metrics, Average: 0.6564
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7701
[CHART] Combined Score: 0.7019 (Similarity: 0.6564, Accuracy: 0.7701)
[ctabganplus] Trial 38: Combined Score: 0.7019 (Similarity: 0.6564, Accuracy: 0.7701) Best Score so far: 0.7248


100%|██████████| 300/300 [01:45<00:00,  2.83it/s]


Finished training in 111.64318466186523  seconds.
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 33/33 valid metrics, Average: 0.6732
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7308
[CHART] Combined Score: 0.6962 (Similarity: 0.6732, Accuracy: 0.7308)
[ctabganplus] Trial 39: Combined Score: 0.6962 (Similarity: 0.6732, Accuracy: 0.7308) Best Score so far: 0.7248


100%|██████████| 350/350 [05:37<00:00,  1.04it/s]


Finished training in 343.0751130580902  seconds.
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 33/33 valid metrics, Average: 0.6409
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7839
[CHART] Combined Score: 0.6981 (Similarity: 0.6409, Accuracy: 0.7839)
[ctabganplus] Trial 40: Combined Score: 0.6981 (Similarity: 0.6409, Accuracy: 0.7839) Best Score so far: 0.7248
  [OK] CTABGAN+: 5 trials in 798.0s
  Best score: 0.7248

[CONTINUE] Optimizing PATE-GAN...
--------------------------------------------------
  Resuming from 35 existing trials
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 33/33 valid metrics, Average: 0.5220
[PRUNED] Trial pruned after similarity calculation (score: 0.5220)
[pategan] Trial 36: Combined Score: 0.5220 (Similarity: 0.0000, Accuracy: 0.0000) Best Score so far: 0.6385
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 33/

### 4.6 Save Best Parameters

In [19]:
# Code Chunk ID: CHUNK_4_6_SAVE
# ============================================================================
# SECTION 4.6 - SAVE BEST PARAMETERS TO CSV
# ============================================================================

from src.utils.parameters import save_best_parameters_to_csv

# Extract studies from manager to globals for saving
for model_name, state in manager.model_states.items():
    if state.study is not None:
        globals()[f"{model_name}_study"] = state.study

# Save all best parameters to CSV for Section 5
save_result = save_best_parameters_to_csv(
    scope=globals(),
    section_number=4,
    dataset_identifier=DATASET_IDENTIFIER
)

print(f"\nParameters saved: {save_result['success']}")
print(f"Files: {save_result.get('files_saved', [])}")

[SAVE] SAVING BEST PARAMETERS FROM SECTION 4
[FOLDER] Target directory: results/alzheimers-disease-data-processed/2026-02-19/Section-4

[CHART] Processing CTGAN parameters...
[OK] Found CTGAN: 10 parameters, score: 0.6580

[CHART] Processing CTAB-GAN parameters...
[OK] Found CTAB-GAN: 2 parameters, score: 0.7269

[CHART] Processing CTAB-GAN+ parameters...
[OK] Found CTAB-GAN+: 2 parameters, score: 0.7248

[CHART] Processing GANerAid parameters...
[WARNING]  GANerAid: Study variable 'ganeraid_study' not found

[CHART] Processing CopulaGAN parameters...
[OK] Found CopulaGAN: 6 parameters, score: 0.6608

[CHART] Processing TVAE parameters...
[OK] Found TVAE: 8 parameters, score: 0.7085

[OK] Parameters saved: results/alzheimers-disease-data-processed/2026-02-19/Section-4/best_parameters.csv
   - Total parameter entries: 32
[OK] Summary saved: results/alzheimers-disease-data-processed/2026-02-19/Section-4/best_parameters_summary.csv
   - Models processed: 5

[SAVE] Parameter saving complet

## 5 Final Model Comparison with Best Parameters

### 5.1 Train All Models with Best Parameters from Section 4

In [20]:
# Code Chunk ID: CHUNK_5_1_BATCH
# ============================================================================
# SECTION 5.1 - BATCH TRAINING WITH BEST PARAMETERS
# ============================================================================

from src.models.batch_training import train_models_batch_with_best_params

# Train all models with best parameters from Section 4
section5_results = train_models_batch_with_best_params(
    data=data,
    target_column=target_column,
    models_to_run=models_to_run,
    categorical_columns=categorical_columns,
    n_samples=len(data),
    random_state=42,
    section_number=4,
    dataset_identifier=DATASET_IDENTIFIER,
    scope=globals(),
    verbose=True
)

# Show summary of created variables
final_vars = [k for k in globals().keys() if k.endswith('_final')]
print(f"\nFinal synthetic data variables: {sorted(final_vars)}")


SECTION 5.1: BATCH TRAINING WITH BEST PARAMETERS
Models to train: 7
Dataset shape: (2149, 33)
Target column: diagnosis
Samples to generate: 2149
Loading parameters from: Section 4

[1/3] Loading best parameters from Section 4...
[LOAD] LOADING BEST PARAMETERS FROM SECTION 4
[FOLDER] Looking for: results/alzheimers-disease-data-processed/2026-02-19/Section-4/best_parameters.csv
[OK] Found parameter CSV file
[OK] Loaded CTGAN: 10 parameters
[OK] Loaded CTAB-GAN: 2 parameters
[OK] Loaded CTAB-GAN+: 2 parameters
[OK] Loaded CopulaGAN: 6 parameters
[OK] Loaded TVAE: 8 parameters

[LOAD] Parameter loading completed!
[SEARCH] Source: CSV file
[CHART] Models loaded: 5
   - ctgan: 10 parameters
   - ctabgan: 2 parameters
   - ctabganplus: 2 parameters
   - copulagan: 6 parameters
   - tvae: 8 parameters
   Parameter source: CSV file
   Models with parameters: 5

[2/3] Training models with optimized parameters...

[1/7] Training CTGAN...
--------------------------------------------------
  Para

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml fo

  Generating 2149 synthetic samples...
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 33/33 valid metrics, Average: 0.6057
[OK] TRTS Evaluation: 2 scenarios, Average: 0.4262
[CHART] Combined Score: 0.5339 (Similarity: 0.6057, Accuracy: 0.4262)
  [OK] CTGAN completed in 90.25s
  Synthetic data shape: (2149, 33)
  Objective score: 0.5339

[2/7] Training TVAE...
--------------------------------------------------
  Parameters loaded: 8
    - epochs: 300
    - batch_size: 128
    - learning_rate: 0.0006
    - embedding_dim: 96
    - l2scale: 0.0000
    ... and 4 more
  Training TVAE...
  Generating 2149 synthetic samples...
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 33/33 valid metrics, Average: 0.5941
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8664
[CHART] Combined Score: 0.7030 (Similarity: 0.5941, Accuracy: 0.8664)
  [OK] TVAE completed in 41.00s
  Synthetic data shape: (2149, 33)
  O

100%|██████████| 400/400 [02:18<00:00,  2.88it/s]


Finished training in 144.32503628730774  seconds.
  Generating 2149 synthetic samples...
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 33/33 valid metrics, Average: 0.6741
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7725
[CHART] Combined Score: 0.7134 (Similarity: 0.6741, Accuracy: 0.7725)
  [OK] CTABGAN completed in 144.55s
  Synthetic data shape: (2149, 33)
  Objective score: 0.7134

[5/7] Training CTABGAN+...
--------------------------------------------------
  Parameters loaded: 2
    - epochs: 350
    - batch_size: 512
    - categorical_columns: ['gender', 'ethnicity', 'educationlevel', 'smoking', 'familyhistoryalzheimers', 'cardiovasculardisease', 'diabetes', 'depression', 'headinjury', 'hypertension', 'memorycomplaints', 'behavioralproblems', 'confusion', 'disorientation', 'personalitychanges', 'difficultycompletingtasks', 'forgetfulness']
    - target_col: diagnosis
  Training CTABGAN+...


100%|██████████| 350/350 [02:03<00:00,  2.83it/s]


Finished training in 130.3289384841919  seconds.
  Generating 2149 synthetic samples...
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 33/33 valid metrics, Average: 0.7000
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7597
[CHART] Combined Score: 0.7239 (Similarity: 0.7000, Accuracy: 0.7597)
  [OK] CTABGAN+ completed in 130.54s
  Synthetic data shape: (2149, 33)
  Objective score: 0.7239

[6/7] Training PATE-GAN...
--------------------------------------------------
  Parameters loaded: 0
    - discrete_columns: ['gender', 'ethnicity', 'educationlevel', 'smoking', 'familyhistoryalzheimers', 'cardiovasculardisease', 'diabetes', 'depression', 'headinjury', 'hypertension', 'memorycomplaints', 'behavioralproblems', 'confusion', 'disorientation', 'personalitychanges', 'difficultycompletingtasks', 'forgetfulness']
  Training PATE-GAN...
  Generating 2149 synthetic samples...
[TARGET] Enhanced objective function using target column: 'diagnosis'


### 5.2 Batch Evaluation of Optimized Models

In [21]:
# Code Chunk ID: CHUNK_5_2_0_A
# ============================================================================
# SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION
# ============================================================================

print("SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION")
print("=" * 80)

from setup import evaluate_section5_optimized_models

try:
    section5_batch_results = evaluate_section5_optimized_models(
        section_number=5,
        scope=globals(),
        target_column=TARGET_COLUMN
    )
    
    print("\n" + "="*80)
    print("SECTION 5.2 OPTIMIZED MODELS BATCH EVALUATION COMPLETED!")
    print("="*80)
    print(f"Models processed: {section5_batch_results['models_processed']}")
    print(f"Results exported to: {section5_batch_results['results_dir']}")
    
except Exception as e:
    print(f"Section 5.2 batch evaluation failed: {e}")
    import traceback
    traceback.print_exc()

SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION
[SEARCH] UNIFIED BATCH EVALUATION - SECTION 5
[INFO] Dataset: alzheimers-disease-data-processed
[INFO] Target column: diagnosis
[INFO] Variable pattern: final
[INFO] Found 7 trained models:
   [OK] CTGAN
   [OK] CTABGAN
   [OK] CTABGANPLUS
   [OK] CopulaGAN
   [OK] TVAE
   [OK] MEDGAN
   [OK] PATEGAN

==================== EVALUATING CTGAN ====================
[SEARCH] CTGAN - COMPREHENSIVE DATA QUALITY EVALUATION
[FOLDER] Output directory: results/alzheimers-disease-data-processed/2026-02-19/Section-5/CTGAN

[1] STATISTICAL SIMILARITY
------------------------------
   [CHART] Average Statistical Similarity: 0.774

[2] PCA COMPARISON ANALYSIS WITH OUTCOME COLOR-CODING
--------------------------------------------------
   [CHART] PCA comparison plot saved: pca_comparison_with_outcome.png
   [CHART] PCA Overall Similarity: 0.017
   [CHART] Explained Variance (PC1, PC2): 0.052, 0.037

[3] DISTRIBUTION SIMILARITY
------------------------------

### 5.3 Final Summary

In [22]:
# Code Chunk ID: CHUNK_5_3_SUMMARY
# ============================================================================
# SECTION 5.3 - FINAL SUMMARY
# ============================================================================

print("=" * 80)
print("SYNTHETIC DATA GENERATION - FINAL SUMMARY")
print("=" * 80)
print(f"\nDataset: {DATASET_IDENTIFIER}")
print(f"Session: {SESSION_TIMESTAMP}")
print(f"\nResults directories:")
print(f"  Section 2 (EDA): {get_results_path(DATASET_IDENTIFIER, 2)}")
print(f"  Section 3 (Demo): {get_results_path(DATASET_IDENTIFIER, 3)}")
print(f"  Section 4 (HPO): {get_results_path(DATASET_IDENTIFIER, 4)}")
print(f"  Section 5 (Final): {get_results_path(DATASET_IDENTIFIER, 5)}")

# Show staged optimization summary
if 'manager' in dir() and manager is not None:
    print(f"\nStaged Optimization Summary:")
    print("-" * 60)
    summary = manager.get_best_params_summary()
    if len(summary) > 0:
        print(summary[['model', 'best_score', 'total_trials', 'status']].to_string(index=False))

# Show final model performance
if 'section5_results' in dir() and section5_results:
    print(f"\nFinal Model Performance (with optimized parameters):")
    print("-" * 60)
    
    scores = []
    for model_name, result in section5_results.items():
        if result['status'] == 'success':
            score = result.get('objective_score', 0)
            time_taken = result.get('training_time', 0)
            scores.append((model_name, score, time_taken))
    
    # Sort by score descending
    scores.sort(key=lambda x: x[1], reverse=True)
    
    for i, (model, score, time_taken) in enumerate(scores, 1):
        print(f"  {i}. {model.upper()}: score={score:.4f}, time={time_taken:.1f}s")
    
    if scores:
        best_model = scores[0][0]
        print(f"\nBest performing model: {best_model.upper()}")
        print(f"Best synthetic data variable: synthetic_{best_model}_final")

print("\n" + "=" * 80)
print("NOTEBOOK COMPLETE")
print("=" * 80)

SYNTHETIC DATA GENERATION - FINAL SUMMARY

Dataset: alzheimers-disease-data-processed
Session: 2026-02-19

Results directories:
  Section 2 (EDA): results/alzheimers-disease-data-processed/2026-02-19/Section-2
  Section 3 (Demo): results/alzheimers-disease-data-processed/2026-02-19/Section-3
  Section 4 (HPO): results/alzheimers-disease-data-processed/2026-02-19/Section-4
  Section 5 (Final): results/alzheimers-disease-data-processed/2026-02-19/Section-5

Staged Optimization Summary:
------------------------------------------------------------
      model  best_score  total_trials    status
      ctgan    0.657953            40 completed
       tvae    0.708459            40 completed
  copulagan    0.660824            40 completed
    ctabgan    0.726908            40 completed
ctabganplus    0.724843            40 completed
    pategan    0.638503            40 completed
     medgan    0.625248            40 completed

Final Model Performance (with optimized parameters):
------------